[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb)

# 🟢 Easy: Gradient Norm Clipping

Implement **gradient norm clipping** — a training stability technique.

### Signature
```python
def clip_grad_norm(parameters, max_norm: float) -> float:
    # Clip gradients in-place so total norm <= max_norm
    # Returns the original (unclipped) total norm
```

### Algorithm
1. Compute total norm: `sqrt(sum(p.grad.norm()^2 for p in parameters))`
2. If total > max_norm: scale all grads by `max_norm / total`
3. Return original total norm

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## What Is Norm?

A norm measures the size of a vector.

In this notebook, `norm()` means the L2 norm, which you can think of as the vector's length in multi-dimensional space.

For a vector `x = [x1, x2, ..., xn]`, its L2 norm is:

`||x||_2 = sqrt(x1^2 + x2^2 + ... + xn^2)`

Here we apply that idea to gradient tensors as well: a gradient tensor can be flattened into a vector, and its norm is the L2 length of that flattened vector.

## One Network, Many Parameters

This function is meant to operate on the parameters of one model.

A single model usually has many parameter tensors, not just one. For example, different layers may each have their own `weight` and `bias`, so `parameters` is a collection of parameter tensors from the same network.

These parameter tensors do not need to have the same shape.

## What Is Total Norm?

Each parameter tensor has its own gradient tensor, and each gradient tensor has its own L2 norm.

The total norm is the global L2 norm across all parameter gradients:

`total_norm = sqrt(||g1||_2^2 + ||g2||_2^2 + ... + ||gn||_2^2)`

You can think of this as flattening every gradient tensor in the whole network, concatenating them into one giant vector, and then measuring the L2 length of that giant vector.

## What Does This Function Clip?

This function clips the parameter gradients of the whole network together.

It does not clip the parameter values themselves, and it does not clip each layer independently. Instead, it computes one global `total_norm` over all parameter gradients in the network, and if that value is too large, it rescales every gradient tensor by the same factor so the global norm becomes at most `max_norm`.

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def clip_grad_norm(parameters, max_norm):
    total_origin_norm = torch.sqrt(sum(p.grad.norm() ** 2 for p in parameters))
    if total_origin_norm > max_norm:
        for p in parameters:
            p.grad *= max_norm / (total_origin_norm + 1e-6)
    return total_origin_norm
    

In [4]:
# 🧪 Debug
p = torch.randn(100, requires_grad=True)
(p * 10).sum().backward()
print('Before:', p.grad.norm().item())
orig = clip_grad_norm([p], max_norm=1.0)
print('After: ', p.grad.norm().item())
print('Original norm:', orig)

Before: 100.0
After:  1.0
Original norm: tensor(100.)


In [5]:
p = torch.randn(5, requires_grad=True)
print(p)
(p * 10).sum().backward()
print(p.grad)
print(p.grad.norm())
print(p.grad.norm().item())

tensor([ 0.4681,  0.8975,  0.7014, -0.0934, -1.1261], requires_grad=True)
tensor([10., 10., 10., 10., 10.])
tensor(22.3607)
22.360679626464844


In [6]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_clipping')


🧪 Testing: Gradient Norm Clipping (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Clips to max_norm (0.8ms)
  ✅ [2/4] Returns original norm (0.5ms)
  ✅ [3/4] No change when norm < max_norm (0.7ms)
  ✅ [4/4] Preserves direction (2.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (4.7ms total)
  Progress saved. Run status() to see your dashboard.

